In [1]:
import os
import json
import math
from pathlib import Path
from collections import Counter
 
import numpy as np
import requests
from PIL import Image
 
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
 
import spacy
 
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

In [2]:
ANNOTATIONS_PATH = "dataset_annotations.json"
SAVE_DIR         = "saved_models"
IMAGE_SIZE       = (224, 224)
BATCH_SIZE       = 32
EPOCHS           = 50
LEARNING_RATE    = 1e-4
RANDOM_STATE     = 42
MIN_LABEL_COUNT  = 100
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
 
nlp = spacy.load("en_core_web_sm")

In [3]:
def load_annotations(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)
 
 
def parse_labels(annotations: dict):
    image_paths   = []
    item_labels   = []
    color_labels  = []
    style_labels  = []
    season_labels = []
 
    for image_path, items in annotations.items():
        image_paths.append(image_path)
 
        items_set   = set()
        colors_set  = set()
        styles_set  = set()
        seasons_set = set()
 
        for item_name, attrs in items.items():
            if attrs.get("item_type"):
                items_set.add(attrs["item_type"])
 
            if attrs.get("color"):
                colors = attrs["color"].lower().replace(",", "").split()
                colors_set.update(colors)
 
            if attrs.get("style"):
                styles_set.add(attrs["style"])
 
            if attrs.get("season"):
                seasons_set.add(attrs["season"])
 
        item_labels.append(list(items_set))
        color_labels.append(list(colors_set))
        style_labels.append(list(styles_set))
        season_labels.append(list(seasons_set))
 
    return image_paths, item_labels, color_labels, style_labels, season_labels

In [4]:
def normalize_labels(texts: list[list[str]]) -> list[list[str]]:
    unique_labels = {label.lower() for text in texts for label in text}
 
    label_map = {}
    for doc in nlp.pipe(unique_labels, batch_size=1000):
        lemmas = " ".join([token.lemma_ for token in doc])
        label_map[doc.text] = lemmas
 
    return [
        [label_map[label.lower()] for label in text]
        for text in texts
    ]
 
 
def filter_labels(label_lists: list[list[str]], min_count: int = MIN_LABEL_COUNT):
    counter = Counter(l for labels in label_lists for l in labels)
    valid   = {l for l, c in counter.items() if c >= min_count}
 
    filtered = [
        [l for l in labels if l in valid]
        for labels in label_lists
    ]
 
    return filtered, valid
 
 
def sync(labels, idx):
    return [labels[i] for i in idx]

In [5]:
train_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
 
val_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
 
 
def load_image(path: str) -> np.ndarray | None:
    if os.path.exists(path):
        try:
            return np.array(Image.open(path).convert("RGB"))
        except Exception as e:
            print(f"  [!] Ошибка чтения {path}: {e}")
            return None
 
    if path.startswith("http"):
        try:
            response = requests.get(path, timeout=10)
            response.raise_for_status()
            return np.array(Image.open(requests.utils.io.BytesIO(response.content)).convert("RGB"))
        except Exception as e:
            print(f"  [!] Ошибка загрузки {path}: {e}")
            return None
 
    print(f"  [!] Файл не найден: {path}")
    return None
 
 
class MultilabelDataset(Dataset):
    """
    Датасет для мультилейбл задачи.
    Y — бинарная матрица (n_samples, n_classes) по всем головам сразу.
    """
 
    def __init__(
        self,
        image_paths: list[str],
        labels_dict: dict[str, np.ndarray],  # {head_name: y_binary}
        transform=None,
    ):
        self.image_paths = image_paths
        self.labels_dict = labels_dict
        self.transform   = transform
 
    def __len__(self):
        return len(self.image_paths)
 
    def __getitem__(self, idx):
        image = load_image(self.image_paths[idx])
 
        if image is None:
            image = np.zeros((*IMAGE_SIZE, 3), dtype=np.uint8)
 
        image = Image.fromarray(image)
        if self.transform:
            image = self.transform(image)
 
        labels = {
            head: torch.tensor(y[idx], dtype=torch.float32)
            for head, y in self.labels_dict.items()
        }
 
        return image, labels

In [6]:
class MultilabelResNet(nn.Module):
    """
    ResNet50 с несколькими головами — по одной на каждый тип лейблов.
    Каждая голова: Linear -> Sigmoid.
    """
 
    def __init__(self, head_sizes: dict[str, int]):
        super().__init__()
 
        backbone    = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        in_features = backbone.fc.in_features
 
        # убираем оригинальный FC
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
 
        # отдельная голова на каждый тип лейблов
        self.heads = nn.ModuleDict({
            head: nn.Linear(in_features, num_classes)
            for head, num_classes in head_sizes.items()
        })
 
    def forward(self, x):
        features = self.backbone(x)           # (batch, 2048, 1, 1)
        features = features.flatten(1)        # (batch, 2048)
 
        return {
            head: self.heads[head](features)  # logits без sigmoid (BCEWithLogitsLoss)
            for head in self.heads
        }

In [7]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
 
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = {h: y.to(DEVICE) for h, y in labels.items()}
 
        optimizer.zero_grad()
 
        outputs = model(images)
 
        # суммируем loss по всем головам
        loss = sum(criterion(outputs[h], labels[h]) for h in outputs)
 
        loss.backward()
        optimizer.step()
 
        total_loss += loss.item()
 
    return total_loss / len(loader)
 
 
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
 
    all_preds  = {h: [] for h in model.heads}
    all_probs  = {h: [] for h in model.heads}
    all_labels = {h: [] for h in model.heads}
 
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = {h: y.to(DEVICE) for h, y in labels.items()}
 
            outputs = model(images)
 
            loss = sum(criterion(outputs[h], labels[h]) for h in outputs)
            total_loss += loss.item()
 
            for h in model.heads:
                probs = torch.sigmoid(outputs[h]).cpu().numpy()
                preds = (probs >= 0.5).astype(int)
 
                all_probs[h].append(probs)
                all_preds[h].append(preds)
                all_labels[h].append(labels[h].cpu().numpy())
 
    # конкатенируем батчи
    all_probs  = {h: np.vstack(v) for h, v in all_probs.items()}
    all_preds  = {h: np.vstack(v) for h, v in all_preds.items()}
    all_labels = {h: np.vstack(v) for h, v in all_labels.items()}
 
    return total_loss / len(loader), all_preds, all_probs, all_labels

In [8]:
def compute_metrics(y_true, y_pred, y_prob) -> dict:
    exact_match = float(np.mean(np.all(y_true == y_pred, axis=1)))
 
    precision_micro = precision_score(y_true, y_pred, average="micro", zero_division=0)
    recall_micro    = recall_score(y_true, y_pred, average="micro", zero_division=0)
    f1_micro        = f1_score(y_true, y_pred, average="micro", zero_division=0)
 
    precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall_macro    = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro        = f1_score(y_true, y_pred, average="macro", zero_division=0)
 
    valid_cols = [i for i in range(y_true.shape[1]) if len(np.unique(y_true[:, i])) > 1]
 
    if valid_cols and y_prob is not None:
        roc_auc = roc_auc_score(
            y_true[:, valid_cols], y_prob[:, valid_cols], average="micro"
        )
        pr_auc = average_precision_score(
            y_true[:, valid_cols], y_prob[:, valid_cols], average="micro"
        )
    else:
        roc_auc = pr_auc = float("nan")
 
    return {
        "exact_match":     exact_match,
        "precision_micro": float(precision_micro),
        "recall_micro":    float(recall_micro),
        "f1_micro":        float(f1_micro),
        "precision_macro": float(precision_macro),
        "recall_macro":    float(recall_macro),
        "f1_macro":        float(f1_macro),
        "roc_auc":         float(roc_auc),
        "pr_auc":          float(pr_auc),
    }
 
 
def print_metrics(metrics: dict) -> None:
    for head_name, m in metrics.items():
        print(f"\n  ▶ Head: {head_name}")
        print(f"  {'-' * 50}")
        print(f"    Exact Match Accuracy : {m['exact_match']:.4f}")
        print(f"    Precision (micro)    : {m['precision_micro']:.4f}")
        print(f"    Recall    (micro)    : {m['recall_micro']:.4f}")
        print(f"    F1        (micro)    : {m['f1_micro']:.4f}")
        print(f"    Precision (macro)    : {m['precision_macro']:.4f}")
        print(f"    Recall    (macro)    : {m['recall_macro']:.4f}")
        print(f"    F1        (macro)    : {m['f1_macro']:.4f}")
        print(f"    ROC AUC   (micro)    : {m['roc_auc']:.4f}")
        print(f"    PR AUC    (micro)    : {m['pr_auc']:.4f}")

In [9]:
def save_model(model, mlb_dict: dict, save_dir: str = SAVE_DIR) -> None:
    """
    Сохраняет веса модели и MLB-кодировщики.
    saved_models/
        model.pt         — веса ResNet
        {head}_mlb.json  — классы каждой головы
    """
    os.makedirs(save_dir, exist_ok=True)
 
    torch.save(model.state_dict(), os.path.join(save_dir, "model.pt"))
    print(f"  Модель сохранена: {save_dir}/model.pt")
 
    for head_name, mlb in mlb_dict.items():
        path = os.path.join(save_dir, f"{head_name}_mlb.json")
        with open(path, "w", encoding="utf-8") as f:
            json.dump(mlb.classes_.tolist(), f, ensure_ascii=False)
        print(f"  MLB сохранён   : {path}")
 
 
def save_metrics(metrics: dict, save_dir: str = SAVE_DIR) -> None:
    """Сохраняет метрики в metrics.json."""
    os.makedirs(save_dir, exist_ok=True)
 
    def clean(val):
        if isinstance(val, float) and (math.isnan(val) or math.isinf(val)):
            return None
        return val
 
    serializable = {
        head: {k: clean(v) for k, v in m.items()}
        for head, m in metrics.items()
    }
 
    path = os.path.join(save_dir, "metrics.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(serializable, f, ensure_ascii=False, indent=2)
 
    print(f"  Метрики сохранены: {path}")
 
 
def load_model(save_dir: str = SAVE_DIR) -> tuple[MultilabelResNet, dict]:
    """
    Загружает модель и MLB-кодировщики.
    Возвращает: (model, {head_name: mlb})
    """
    mlb_dict   = {}
    head_sizes = {}
 
    for filename in os.listdir(save_dir):
        if not filename.endswith("_mlb.json"):
            continue
 
        head_name = filename.replace("_mlb.json", "")
        path      = os.path.join(save_dir, filename)
 
        with open(path, "r", encoding="utf-8") as f:
            classes = json.load(f)
 
        mlb = MultiLabelBinarizer()
        mlb.classes_ = np.array(classes)
        mlb_dict[head_name]   = mlb
        head_sizes[head_name] = len(classes)
 
    model = MultilabelResNet(head_sizes).to(DEVICE)
    model.load_state_dict(torch.load(os.path.join(save_dir, "model.pt"), map_location=DEVICE))
    model.eval()
 
    print(f"  Модель загружена из {save_dir}")
    return model, mlb_dict
 
 
def load_metrics(save_dir: str = SAVE_DIR) -> dict:
    path = os.path.join(save_dir, "metrics.json")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [10]:
print(f"  Устройство: {DEVICE}")
 
print("\n── 1. Загрузка и подготовка лейблов ───────────────────")
annotations = load_annotations(ANNOTATIONS_PATH)
image_paths, item_labels, color_labels, style_labels, season_labels = parse_labels(annotations)

item_labels   = normalize_labels(item_labels)
color_labels  = normalize_labels(color_labels)
style_labels  = normalize_labels(style_labels)
season_labels = normalize_labels(season_labels)

item_labels,   _ = filter_labels(item_labels)
color_labels,  _ = filter_labels(color_labels)
style_labels,  _ = filter_labels(style_labels)
season_labels, _ = filter_labels(season_labels)

print("\n── 2. Бинаризация лейблов ──────────────────────────────")
heads = {
    "item":   {"labels": item_labels,   "mlb": MultiLabelBinarizer()},
    "color":  {"labels": color_labels,  "mlb": MultiLabelBinarizer()},
    "style":  {"labels": style_labels,  "mlb": MultiLabelBinarizer()},
    "season": {"labels": season_labels, "mlb": MultiLabelBinarizer()},
}
Y_dict = {}
for head_name, data in heads.items():
    Y_dict[head_name] = data["mlb"].fit_transform(data["labels"])
    print(f"  {head_name}: {len(data['mlb'].classes_)} классов")

print("\n── 3. Разбиение на train/test (80/20) ──────────────────")
Y_all = np.hstack(list(Y_dict.values()))
n     = len(image_paths)

mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(mskf.split(np.zeros(n), Y_all))

train_paths = [image_paths[i] for i in train_idx]
test_paths  = [image_paths[i] for i in test_idx]

train_labels = {h: Y_dict[h][train_idx] for h in heads}
test_labels  = {h: Y_dict[h][test_idx]  for h in heads}

print(f"  Train: {len(train_idx)}  |  Test: {len(test_idx)}")

print("\n── 4. Подготовка DataLoader ────────────────────────────")
train_dataset = MultilabelDataset(train_paths, train_labels, transform=train_transforms)
test_dataset  = MultilabelDataset(test_paths,  test_labels,  transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("\n── 5. Инициализация модели ─────────────────────────────")
head_sizes = {h: len(data["mlb"].classes_) for h, data in heads.items()}
model      = MultilabelResNet(head_sizes).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("\n── 6. Обучение ─────────────────────────────────────────")
best_f1       = 0.0
best_metrics  = None

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, all_preds, all_probs, all_labels = eval_epoch(model, test_loader, criterion)

    # метрики по каждой голове
    epoch_metrics = {
        h: compute_metrics(all_labels[h], all_preds[h], all_probs[h])
        for h in heads
    }

    mean_f1 = np.mean([m["f1_micro"] for m in epoch_metrics.values()])

    print(
        f"  Epoch {epoch:02d}/{EPOCHS}"
        f"  train_loss={train_loss:.4f}"
        f"  val_loss={val_loss:.4f}"
        f"  mean_f1_micro={mean_f1:.4f}"
    )

    # сохраняем лучшую модель
    if mean_f1 > best_f1:
        best_f1      = mean_f1
        best_metrics = epoch_metrics
        save_model(model, {h: data["mlb"] for h, data in heads.items()})
        print(f"  ✓ Лучшая модель сохранена (f1={best_f1:.4f})")

print("\n── 7. Итоговые метрики лучшей модели ───────────────────")
print_metrics(best_metrics)
save_metrics(best_metrics)

  Устройство: cuda

── 1. Загрузка и подготовка лейблов ───────────────────

── 2. Бинаризация лейблов ──────────────────────────────
  item: 23 классов
  color: 14 классов
  style: 11 классов
  season: 3 классов

── 3. Разбиение на train/test (80/20) ──────────────────
  Train: 11998  |  Test: 3002

── 4. Подготовка DataLoader ────────────────────────────

── 5. Инициализация модели ─────────────────────────────

── 6. Обучение ─────────────────────────────────────────
  [!] Файл не найден: images_for_training/Charming_Boucl&eacute;_Dress_img_00000036.jpg
  [!] Файл не найден: images_for_training/Oversized_Boucl&eacute;_Moto_Jacket_img_00000007.jpg
  [!] Файл не найден: images_for_training/Marled_Boucl&eacute;_Moto_Jacket_img_00000035.jpg
  [!] Файл не найден: images_for_training/Ombr&eacute;_Tank_img_00000070.jpg
  [!] Файл не найден: images_for_training/Macram&eacute;-Back_LA_Tank_img_00000020.jpg
  [!] Файл не найден: images_for_training/Distressed_Open-Knit_Ombr&eacute;_Sweater_im